In [1]:
import re
import pandas as pd
from datetime import datetime, timezone
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import json

In [2]:
# ============================================================
# CONFIG
# ============================================================

PLAYER = "Leo Carlsson"
PLAYER_URL = "https://puckpedia.com/player/leo-carlsson"

In [3]:
# ============================================================
# HELPERS
# ============================================================

def clean_text(value):
    if value is None:
        return None
    value = re.sub(r"\s+", " ", value).strip()
    return value or None


def parse_money(value):
    if not value:
        return None

    text = value.replace("$", "").replace(",", "").strip().upper()

    match = re.search(r"([\d.]+)\s*([KMB])?", text)
    if not match:
        return None

    number = float(match.group(1))
    multiplier = {
        "K": 1_000,
        "M": 1_000_000,
        "B": 1_000_000_000,
        None: 1,
    }[match.group(2)]

    return int(number * multiplier)

def parse_int(value):
    if value is None:
        return None

    match = re.search(r"\d+", str(value).replace(",", ""))
    return int(match.group()) if match else None

In [4]:
# ============================================================
# FETCH PAGE
# ============================================================

async def get_player_html(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        page = await browser.new_page(
            viewport={"width": 1600, "height": 1200}
        )

        await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000,
        )

        await page.wait_for_timeout(1500)

        html = await page.content()

        await browser.close()

        return html

html = await get_player_html(PLAYER_URL)

print(f"HTML length: {len(html):,}")

HTML length: 27,555


In [13]:
def parse_player_detail(html, url):
    soup = BeautifulSoup(html, "html.parser")

    row = {
        "player": None,
        "player_url": url,
        "leadership_role": None,
        "sweater_number": None,
        "age": None,
        "position": None,
        "shoots_catches": None,
        "height": None,
        "height_inches": None,
        "weight_lbs": None,
        "depth_chart_position": None,
        "depth_chart_line": None,
        "drafted": False,
        "draft_round": None,
        "draft_pick": None,
        "draft_year": None,
        "agent": None,
        "birthdate": None,
        "birthplace": None,
        "nationality": None,
        "ufa_year": None,
        "elc_age": None,
        "waivers_eligibility": None,
        "estimated_career_earnings": None,
        "current_contract": False,
        "current_cap_hit": None,
        "current_contract_year": None,
        "current_contract_term": None,
        "current_contract_start_season": None,
        "current_contract_end_season": None,
        "current_expiry_status": None,
        "current_expiry_year": None,
        "current_expiry_age": None,
        "source_url": url,
        "scrape_datetime": datetime.now(timezone.utc),
    }

    # ---------------------------------------------------------
    # JSON-LD
    # ---------------------------------------------------------
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
        except (json.JSONDecodeError, TypeError):
            continue

        if data.get("@type") != "SportsTeam":
            continue

        role = data.get("member", {})
        person = role.get("member", {})

        if person.get("@type") != "Person":
            continue

        row["player"] = person.get("name")
        row["sweater_number"] = parse_int(role.get("numberedPosition"))

        role_name = role.get("roleName")
        position_map = {
            "Center": "C",
            "Left Wing": "LW",
            "Right Wing": "RW",
            "Defense": "D",
            "Defence": "D",
            "Goalie": "G",
            "Goaltender": "G",
        }
        row["position"] = position_map.get(role_name, role_name)

        row["birthdate"] = person.get("birthDate")
        row["nationality"] = person.get("nationality")

        height = person.get("height", {})
        weight = person.get("weight", {})
        earnings = person.get("netWorth", {})

        row["height_inches"] = parse_int(height.get("value"))
        row["weight_lbs"] = parse_int(weight.get("value"))

        if earnings.get("value"):
            row["estimated_career_earnings"] = parse_money(
                str(earnings.get("value"))
            )

        if row["height_inches"]:
            feet, inches = divmod(row["height_inches"], 12)
            row["height"] = f"{feet}'{inches}\""

        if row["birthdate"]:
            try:
                dob = datetime.strptime(
                    row["birthdate"], "%Y-%m-%d"
                ).date()

                today = datetime.now().date()

                row["age"] = (
                    today.year
                    - dob.year
                    - (
                        (today.month, today.day)
                        < (dob.month, dob.day)
                    )
                )
            except ValueError:
                pass

        break

    # ---------------------------------------------------------
    # PROFILE STATS
    # # / AGE / POS / SHOT / HEIGHT / WEIGHT
    # ---------------------------------------------------------
    for label in soup.select(".pp_subset"):
        key = clean_text(label.get_text(" ", strip=True))
        parent = label.parent

        if not parent:
            continue

        value_element = parent.select_one(".statsrow_val")
        if not value_element:
            continue

        value = clean_text(value_element.get_text(" ", strip=True))

        if not value:
            continue

        key_lower = key.lower() if key else ""

        if key_lower == "#":
            row["sweater_number"] = parse_int(value)

        elif key_lower == "age":
            row["age"] = parse_int(value)

        elif key_lower == "pos":
            row["position"] = value.upper()

        elif key_lower in {"shot", "catches"}:
            row["shoots_catches"] = value.upper()

        elif key_lower == "h":
            row["height"] = value

            match = re.search(r"(\d+)['′]\s*(\d+)", value)
            if match:
                row["height_inches"] = (
                    int(match.group(1)) * 12
                    + int(match.group(2))
                )

        elif key_lower == "w":
            row["weight_lbs"] = parse_int(value)

    # ---------------------------------------------------------
    # DEPTH CHART
    # Example: C1 / 1st Line
    # ---------------------------------------------------------
    depth = soup.select_one(".pp_dc")

    if depth:
        chip = depth.select_one(".pp_dc_chip")
        value = depth.select_one(".pp_dc_value")

        if chip:
            chip_text = clean_text(chip.get_text(" ", strip=True))

            if chip_text:
                match = re.match(r"([A-Za-z]+)(\d+)", chip_text)

                if match:
                    row["depth_chart_position"] = match.group(1).upper()
                    row["depth_chart_line"] = int(match.group(2))
                else:
                    row["depth_chart_position"] = chip_text

        if value and row["depth_chart_line"] is None:
            line_text = clean_text(value.get_text(" ", strip=True))

            if line_text:
                row["depth_chart_line"] = parse_int(line_text)

    # ---------------------------------------------------------
    # MICRO ROWS
    # UFA / ELC / WAIVERS / CAREER EARNINGS etc.
    # ---------------------------------------------------------
    for micro in soup.select(".micro_row"):
        label_element = micro.select_one(".micro_label")
        value_element = micro.select_one(".micro_value")

        if not label_element or not value_element:
            continue

        label = clean_text(label_element.get_text(" ", strip=True))
        value = clean_text(value_element.get_text(" ", strip=True))

        if not label or not value:
            continue

        label_lower = label.lower()

        if "ufa year" in label_lower:
            row["ufa_year"] = parse_int(value)

        elif "elc age" in label_lower:
            row["elc_age"] = parse_int(value)

        elif "waivers eligibility" in label_lower:
            row["waivers_eligibility"] = value

        elif "career earnings" in label_lower:
            money = parse_money(value)

            if money is not None:
                row["estimated_career_earnings"] = money

    # ---------------------------------------------------------
    # DRAFT
    # ---------------------------------------------------------
    draft_text = None

    for element in soup.find_all(string=re.compile(r"Draft", re.I)):
        parent = element.parent

        if not parent:
            continue

        container = parent.parent

        if not container:
            continue

        text = clean_text(container.get_text(" ", strip=True))

        if text and re.search(r"\bRound\b|\bPick\b|\b20\d{2}\b", text, re.I):
            draft_text = text
            break

    if draft_text:
        row["drafted"] = True

        match = re.search(r"Round\s*(\d+)", draft_text, re.I)
        if match:
            row["draft_round"] = int(match.group(1))

        match = re.search(r"Pick\s*(\d+)", draft_text, re.I)
        if match:
            row["draft_pick"] = int(match.group(1))

        match = re.search(r"\b(20\d{2})\b", draft_text)
        if match:
            row["draft_year"] = int(match.group(1))

    # ---------------------------------------------------------
    # LABEL/VALUE EXTRACTION
    # Useful for Agent / Born / Birthplace / Draft
    # ---------------------------------------------------------
    labels = soup.find_all(
        string=re.compile(
            r"^(Agent|Born|Birthplace|Drafted|Draft)$",
            re.I,
        )
    )

    for label_node in labels:
        label = clean_text(str(label_node))

        if not label:
            continue

        parent = label_node.parent

        if not parent:
            continue

        container = parent.parent

        if not container:
            continue

        text = clean_text(container.get_text(" ", strip=True))

        if not text:
            continue

        value = re.sub(
            rf"^{re.escape(label)}\s*:?\s*",
            "",
            text,
            flags=re.I,
        ).strip()

        if label.lower() == "agent" and value:
            row["agent"] = value

        elif label.lower() in {"born", "birthplace"} and value:
            if not row["birthplace"]:
                row["birthplace"] = value

    # ---------------------------------------------------------
    # FALLBACK AGENT SEARCH
    # ---------------------------------------------------------
    if not row["agent"]:
        agent_match = re.search(
            r"\bAgent\s*:?\s*([A-Z][A-Za-z.'’-]+(?:\s+[A-Z][A-Za-z.'’-]+){1,3})",
            page_text,
        )

        if agent_match:
            row["agent"] = clean_text(agent_match.group(1))

    # ---------------------------------------------------------
    # LEADERSHIP
    # ---------------------------------------------------------
    for text in soup.stripped_strings:
        value = clean_text(text)

        if value in {"Captain", "A. Captain", "Alternate Captain"}:
            row["leadership_role"] = value
            break

    # ---------------------------------------------------------
    # CURRENT CONTRACT
    # Locate the actual Current Contract heading.
    # ---------------------------------------------------------
    current_contract_heading = None

    for heading in soup.find_all(["h2", "h3", "h4"]):
        if "current contract" in heading.get_text(
            " ", strip=True
        ).lower():
            current_contract_heading = heading
            break

    if current_contract_heading:
        row["current_contract"] = True

        # Use the surrounding contract section.
        contract_section = current_contract_heading.find_next("div")

        # PuckPedia's current contract block is followed by
        # contract data before the next major separator/section.
        contract_parts = []

        node = current_contract_heading

        for _ in range(80):
            node = node.find_next()

            if node is None:
                break

            if (
                node.name in {"h2", "h3"}
                and node is not current_contract_heading
            ):
                break

            if node.name:
                text = clean_text(
                    node.get_text(" ", strip=True)
                )

                if text:
                    contract_parts.append(text)

        contract_text = " ".join(contract_parts)

        # Contract season range
        season_match = re.search(
            r"\b(20\d{2})-(20\d{2})\b",
            contract_text
        )

        if season_match:
            start_year = int(season_match.group(1))
            end_year = int(season_match.group(2))

            term = end_year - start_year

            row["current_contract_term"] = term
            row["current_contract_start_season"] = f"{start_year}-{str(start_year + 1)[-2:]}"
            row["current_contract_end_season"] = f"{end_year - 1}-{str(end_year)[-2:]}"

        # Cap hit
        cap_match = re.search(
            r"Cap Hit\s+\$([\d,]+(?:\.\d+)?)",
            contract_text,
            re.I,
        )

        if cap_match:
            row["current_cap_hit"] = parse_money(
                "$" + cap_match.group(1)
            )

        # Expiry
        expiry_match = re.search(
            r"Expiry Status\s+(UFA|RFA|10\.2\(c\)|10\.2c)"
            r"\s+(\d{4})\s+Age\s+(\d+)",
            contract_text,
            re.I,
        )

        if expiry_match:
            row["current_expiry_status"] = expiry_match.group(1).upper()
            row["current_expiry_year"] = int(expiry_match.group(2))
            row["current_expiry_age"] = int(expiry_match.group(3))

        if row["current_contract_start_season"] and row["current_contract_term"]:
            today = datetime.now().date()

            season_start_year = today.year if today.month >= 7 else today.year - 1
            contract_start_year = int(row["current_contract_start_season"][:4])

            contract_year = season_start_year - contract_start_year + 1

            if 1 <= contract_year <= row["current_contract_term"]:
                row["current_contract_year"] = contract_year

        # Determine current contract year from season range.
        if (
            row["current_contract_start_season"]
            and row["current_contract_end_season"]
        ):
            today = datetime.now().date()

            # NHL season year changes around July.
            season_start_year = (
                today.year
                if today.month >= 7
                else today.year - 1
            )

            contract_start_year = int(
                row["current_contract_start_season"][:4]
            )

            contract_year = (
                season_start_year
                - contract_start_year
                + 1
            )

            if (
                row["current_contract_term"]
                and 1 <= contract_year <= row["current_contract_term"]
            ):
                row["current_contract_year"] = contract_year

    return row

In [14]:
row = parse_player_detail(
    html=html,
    url=PLAYER_URL
)

df_detail = pd.DataFrame([row])

pd.set_option("display.max_columns", None)

display(df_detail.T)

,0
player,Leo Carlsson
player_url,https://puckpedia.com/player/leo-carlsson
leadership_role,A. Captain
sweater_number,91
age,21
position,C
shoots_catches,L
height,"6'3"""
height_inches,75
weight_lbs,207


In [15]:
async def get_player_html(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        context = await browser.new_context(
            viewport={"width": 1600, "height": 1200},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
        )

        page = await context.new_page()

        response = await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000,
        )

        print(f"HTTP status: {response.status if response else 'None'}")
        print(f"Final URL: {page.url}")

        # Wait until the actual PuckPedia player data has rendered
        try:
            await page.wait_for_selector(
                'script[type="application/ld+json"]',
                timeout=30000,
            )
        except:
            pass

        await page.wait_for_timeout(3000)

        html = await page.content()

        print(f"HTML length: {len(html):,}")
        print(f"Leo found: {'Leo Carlsson' in html}")
        print(f"JSON-LD found: {'application/ld+json' in html}")

        await browser.close()

        return html

In [16]:
PLAYER_URL = "https://puckpedia.com/player/leo-carlsson"

html = await get_player_html(PLAYER_URL)

HTTP status: 200
Final URL: https://puckpedia.com/player/leo-carlsson
HTML length: 1,019,103
Leo found: True
JSON-LD found: True
